# ДЗ 1. PyTorch: тензоры и автоматическое дифференцирование

Домашнее задание к теме «Введение в PyTorch, автоматическое дифференцирование».
Предполагается, что вы уже прошли материал лекции (`pytorch_basics.ipynb`): базовые операции
с тензорами, линейную алгебру, `backward()`, `torch.autograd.functional.jacobian/hessian`,
работу с Якобианом и Гессианом.

**Как устроено задание.** Два уровня сложности: в первом — три задачи, во втором — одна,
более комплексная. Задачи независимы, можно решать в любом порядке. Ни одна задача не требует
GPU и не займёт больше нескольких минут вычислений на CPU — узкое место везде в том, чтобы
правильно написать код, а не в том, чтобы дождаться обучения.

- **Уровень 1 (базовый).** Уверенное использование того, что было на лекции.
- **Уровень 2 (средний).** Собственный `autograd.Function`, работа с численной устойчивостью.

В каждой задаче есть блок `assert`-проверок — это не полноценные тесты, а быстрый способ
убедиться, что вы не ошиблись в очевидном месте. Проверки проходят — не значит, что решение
идеально; не проходят — значит, точно есть ошибка.


In [1]:
import torch
import numpy as np
import math
torch.manual_seed(0)

## Уровень 1. Попроще

### Задача 1.1. Тензорная разминка

Реализуйте три функции, используя **только** тензорные операции PyTorch (без циклов по элементам
на Python — весь смысл в том, чтобы привыкнуть мыслить «вектором», а не поэлементно).

1. `col_normalize(X)` — для матрицы `X` формы `(n, d)` верните матрицу, в которой каждый
   **столбец** центрирован (вычтено среднее по столбцу) и поделён на своё стандартное отклонение.
2. `pairwise_cosine_sim(X, Y)` — для `X` формы `(n, d)` и `Y` формы `(m, d)` верните матрицу
   `(n, m)` попарных косинусных сходств между строками `X` и строками `Y`. Подсказка: косинусное
   сходство — это скалярное произведение **нормированных** векторов, поэтому всё считается через
   одно матричное умножение нормированных матриц и broadcasting при делении на нормы — без
   единого Python-цикла.
3. `one_hot_with_ignore(labels, n_classes, ignore_index=-1)` — для целочисленного тензора меток
   формы `(n,)` верните one-hot матрицу формы `(n, n_classes)`, в которой строки с
   `labels[i] == ignore_index` должны быть полностью нулевыми (а не one-hot). Посмотрите на
   `torch.scatter_` и булеву индексацию.

**Проверьте себя:** сравните `pairwise_cosine_sim` с эталонной реализацией через
`torch.nn.functional.cosine_similarity` (с `unsqueeze`, чтобы получить broadcasting сразу по двум
осям) — они должны совпадать с точностью до `1e-5`.


In [2]:
import torch.nn.functional as F
import pandas as pd


def col_normalize(X):
    std, mean = torch.std_mean(X, dim=0)
    return (X - mean) / std


def pairwise_cosine_sim(X, Y, eps=1e-8):
    norms = X.norm(dim=1, keepdim=True) * Y.norm(dim=1)
    return X @ Y.T / norms.clamp_min(eps)


def one_hot_with_ignore(labels, n_classes, ignore_index=-1):
    hit = labels[:, None] == torch.arange(n_classes, device=labels.device)
    return (hit & (labels != ignore_index)[:, None]).float()


X = torch.randn(50, 7) * torch.arange(1, 8) + 3
Z = col_normalize(X)
assert torch.allclose(Z.mean(0), torch.zeros(7), atol=1e-5)
assert torch.allclose(Z.std(0), torch.ones(7), atol=1e-5)

A, B = torch.randn(40, 16), torch.randn(25, 16)
ref = F.cosine_similarity(A.unsqueeze(1), B.unsqueeze(0), dim=-1)
assert torch.allclose(pairwise_cosine_sim(A, B), ref, atol=1e-5)
assert torch.allclose(pairwise_cosine_sim(A, A).diagonal(), torch.ones(40), atol=1e-5)

labels = torch.tensor([2, -1, 0, 3, -1, 1])
oh = one_hot_with_ignore(labels, 4)
ref = F.one_hot(labels.clamp_min(0), 4) * (labels != -1)[:, None]
assert oh.shape == (6, 4) and torch.equal(oh, ref.float())
assert one_hot_with_ignore(labels, 4, ignore_index=0)[2].sum() == 0

`one_hot_with_ignore` сделан без `scatter_`: сравнение меток с `arange(n_classes)` через broadcasting сразу даёт булеву матрицу `(n, n_classes)`. Метка `-1` ни с одним классом не совпадает и так, но маска по `ignore_index` всё равно нужна, чтобы функция работала и при `ignore_index`, попадающем в диапазон классов (последний assert)

### Задача 1.2. Логистическая регрессия на чистом autograd

Возьмите датасет `sklearn.datasets.load_breast_cancer` (30 числовых признаков, бинарная
классификация: доброкачественная/злокачественная опухоль). Обучите логистическую регрессию
$p(y=1\mid x) = \sigma(w^\top x + b)$, **не используя** `torch.nn` и `torch.optim` — только
тензоры, `requires_grad=True`, ручной шаг градиентного спуска внутри `with torch.no_grad():`,
обнуление градиентов, и функцию потерь — бинарную кросс-энтропию, реализованную вручную (не
пользуйтесь `torch.nn.functional.binary_cross_entropy` и её вариантами).

Стандартизируйте признаки (`StandardScaler` или руками через `mean`/`std`) — без этого
градиентный спуск будет сходиться на порядки медленнее из-за разных масштабов признаков.

**Проверьте себя:** обучите `sklearn.linear_model.LogisticRegression` на тех же
стандартизированных признаках (чтобы сравнение было честным, отключите регуляризацию:
`LogisticRegression(penalty=None)`, а в старых версиях sklearn — `C=1e6`) и сравните полученные
веса, а также итоговую accuracy на обучающей выборке — они должны быть близки.


In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

X_raw, y_raw = load_breast_cancer(return_X_y=True)
X = torch.tensor(X_raw)
X = (X - X.mean(0)) / X.std(0, correction=0)
y = torch.tensor(y_raw, dtype=torch.float64)


def bce_with_logits(z, y):
    return (torch.logaddexp(torch.zeros_like(z), z) - y * z).mean()


def fit_logreg(X, y, lam=0.0, steps=20000):
    Xb = torch.cat([X, torch.ones(len(X), 1, dtype=X.dtype)], dim=1)
    lr = 1 / (torch.linalg.matrix_norm(Xb, 2) ** 2 / (4 * len(X)) + lam)
    w = torch.zeros(X.shape[1], dtype=X.dtype, requires_grad=True)
    b = torch.zeros((), dtype=X.dtype, requires_grad=True)
    for _ in range(steps):
        loss = bce_with_logits(X @ w + b, y) + lam / 2 * w @ w
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
            w.grad.zero_()
            b.grad.zero_()
    return w.detach(), b.detach()


def compare(w, b, sk):
    w_sk = torch.tensor(sk.coef_[0])
    return {
        'acc torch': ((X @ w + b > 0) == y.bool()).double().mean().item(),
        'acc sklearn': sk.score(X.numpy(), y_raw),
        '||w|| torch': w.norm().item(),
        '||w|| sklearn': w_sk.norm().item(),
        'cos(w, w_sk)': F.cosine_similarity(w, w_sk, dim=0).item(),
        'max |w - w_sk|': (w - w_sk).abs().max().item(),
    }


C = 1.0
w0, b0 = fit_logreg(X, y)
w1, b1 = fit_logreg(X, y, lam=1 / (C * len(X)))
sk0 = LogisticRegression(C=1e12, max_iter=100_000).fit(X.numpy(), y_raw)
sk1 = LogisticRegression(C=C, max_iter=100_000, tol=1e-10).fit(X.numpy(), y_raw)

res = pd.DataFrame([compare(w0, b0, sk0), compare(w1, b1, sk1)], index=['без регуляризации', f'L2, C={C}'])
assert res.loc[f'L2, C={C}', 'max |w - w_sk|'] < 1e-4
assert (res['acc torch'] - res['acc sklearn']).abs().max() < 0.02
res.round(4)

,acc torch,acc sklearn,||w|| torch,||w|| sklearn,"cos(w, w_sk)",max |w - w_sk|
без регуляризации,0.9895,1.0000,10.8350,59452.3635,0.4201,36719.9678
"L2, C=1.0",0.9877,0.9877,3.8416,3.8416,1.0000,0.0000


Стандартизованный breast_cancer линейно разделим: sklearn без регуляризации получает accuracy 1.0 на train. Для разделимой выборки минимума логлосса не существует, лосс стремится к нулю при ‖w‖ → ∞, поэтому оба решения без регуляризации уходят в бесконечность, просто с разной скоростью (у GD норма растёт примерно как log t, lbfgs за пару сотен итераций разгоняется до десятков тысяч). Поэлементно сравнивать такие веса бессмысленно, близки только accuracy. «Без регуляризации» в sklearn задано как C=1e12: это то же самое, но без предупреждений, `penalty=None` в свежих версиях объявлен устаревшим

Чтобы веса можно было сравнить честно, в оба метода добавлена одна и та же L2. sklearn минимизирует ½‖w‖² + C·Σℓᵢ, после деления на C·n получается средний логлосс + ‖w‖²/(2Cn), то есть λ = 1/(Cn). Задача становится строго выпуклой, минимум единственный, и веса совпадают до точности оптимизатора sklearn

Шаг GD не подбирался: lr = 1/L, где L = ‖[X, 1]‖²₂ / (4n) + λ, константа Липшица градиента (σ' ≤ 1/4). С таким шагом спуск гарантированно не расходится

BCE записан через логиты, log(1 + eᶻ) − y·z. Это тождественно −y·log σ(z) − (1 − y)·log(1 − σ(z)), но без log(0) на уверенных предсказаниях, что для разделимых данных как раз актуально

### Задача 1.3. Проверка градиентов и охота за багом

**Часть А.** Реализуйте две функции численного градиента функции `f: R^n -> R`:

- `numerical_grad_forward(f, x, eps)` — односторонняя (прямая) разность:
  $\dfrac{\partial f}{\partial x_i} \approx \dfrac{f(x+\epsilon e_i) - f(x)}{\epsilon}$;
- `numerical_grad_central(f, x, eps)` — центральная разность:
  $\dfrac{\partial f}{\partial x_i} \approx \dfrac{f(x+\epsilon e_i) - f(x-\epsilon e_i)}{2\epsilon}$.

Сравните обе с autograd-градиентом на функции $f(x) = \sum_i x_i^3 - \cos(x_1)\cdot x_2$ (или
придумайте свою) для нескольких значений `eps` (например, $10^{-2}, 10^{-4}, 10^{-6}, 10^{-8}$) и
постройте таблицу или график ошибки каждого метода в зависимости от `eps`. Кратко объясните в
markdown-ячейке: почему ошибка центральной разности убывает быстрее (теоретически как
$O(\epsilon^2)$ против $O(\epsilon)$ у односторонней), и почему при слишком маленьком `eps` ошибка
обоих методов снова начинает расти.

**Часть Б.** Ниже — код с сознательно внесённой ошибкой в такой сравнительной проверке (баг
может быть в самой функции, а не в проверке). Найдите ошибку, объясните словами, в чём она, и
исправьте.

```python
def buggy_grad_check():
    def f(x):
        return torch.sin(x[0]) * x[1]**2 + x[0] * x[2]

    x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
    y = f(x)
    y.backward()
    grad_auto = x.grad

    # "ручной" аналитический градиент, вычисленный на бумаге (с ошибкой!)
    grad_manual = torch.tensor([
        torch.cos(x[0]) * x[1]**2,
        2 * torch.sin(x[0]) * x[1],
        x[0],
    ])
    print(torch.allclose(grad_auto, grad_manual, atol=1e-4))
```


In [4]:
from torch.func import vmap


def numerical_grad_forward(f, x, eps):
    shifts = eps * torch.eye(len(x), dtype=x.dtype)
    return (vmap(f)(x + shifts) - f(x)) / eps


def numerical_grad_central(f, x, eps):
    shifts = eps * torch.eye(len(x), dtype=x.dtype)
    return (vmap(f)(x + shifts) - vmap(f)(x - shifts)) / (2 * eps)


def f(x):
    return (x ** 3).sum() - torch.cos(x[0]) * x[1]


powers = range(1, 12)
eps_grid = [10.0 ** -k for k in powers]
errors = {}
for dtype in (torch.float32, torch.float64):
    x = torch.tensor([0.7, -1.3, 2.1, 0.4], dtype=dtype, requires_grad=True)
    f(x).backward()
    x0, g = x.detach(), x.grad
    name = str(dtype).split('.')[-1]
    errors[f'forward {name}'] = [(numerical_grad_forward(f, x0, e) - g).abs().max().item() for e in eps_grid]
    errors[f'central {name}'] = [(numerical_grad_central(f, x0, e) - g).abs().max().item() for e in eps_grid]

pd.DataFrame(errors, index=pd.Index([f'1e-{k}' for k in powers], name='eps')).map('{:.1e}'.format)

,forward float32,central float32,forward float64,central float64
eps,,,,
1e-1,6.4e-01,1.1e-02,6.4e-01,1.1e-02
1e-2,6.3e-02,1.5e-04,6.3e-02,1.1e-04
1e-3,5.1e-03,1.1e-03,6.3e-03,1.1e-06
1e-4,1.2e-02,1.7e-02,6.3e-04,1.1e-08
1e-5,3.5e-02,2.6e-02,6.3e-05,1.5e-10
1e-6,8.3e-01,8.3e-01,6.3e-06,2.2e-09
1e-7,1.3e+01,1.3e+01,6.2e-07,1.6e-08
1e-8,1.3e+01,1.3e+01,1.4e-07,1.5e-07
1e-9,1.3e+01,1.3e+01,1.1e-06,1.1e-06


Численные градиенты посчитаны без цикла по координатам: сдвиги ε·eᵢ для всех i лежат в строках ε·I, и `vmap` прогоняет f по всем сдвинутым точкам за один вызов

По Тейлору f(x ± εeᵢ) = f ± ε∂ᵢf + ε²∂ᵢ²f/2 ± ε³∂ᵢ³f/6 + …

В односторонней разности после деления на ε остаётся член ε∂ᵢ²f/2, отсюда O(ε). В центральной при вычитании f(x − εeᵢ) все чётные члены сокращаются, первый ненулевой остаток ε²∂ᵢ³f/6, отсюда O(ε²)

При малом ε начинает доминировать округление. f вычисляется с относительной погрешностью порядка машинного эпсилона u, при вычитании двух почти равных значений значащие разряды теряются, а затем результат ещё и делится на ε, так что эта часть ошибки ~ u·|f|/ε и растёт при уменьшении ε. Сумма двух ошибок минимальна при ε порядка √u для прямой разности и ∛u для центральной (с поправкой на масштаб f и её производных). В таблице для float64 (u ≈ 1e-16) минимумы ровно на 1e-8 и 1e-5, для float32 (u ≈ 1e-7) на 1e-3 и 1e-2, и центральная разность выигрывает у прямой на порядки только в своей оптимальной зоне. В float32 на самых маленьких ε шаг вообще меньше расстояния между соседними представимыми числами, x + ε == x, численная производная обнуляется и ошибка равна самому градиенту

**Часть Б.** Ошибка в ручном градиенте по x₀. Слагаемое x₀·x₂ тоже зависит от x₀, и его производная x₂ потеряна: правильно ∂f/∂x₀ = cos(x₀)·x₁² + x₂. В точке (1, 2, 3) это 5.16 против 2.16, поэтому `allclose` даёт False, а autograd здесь прав. Остальные две компоненты посчитаны верно

Попутно ещё два момента, которые не ломают проверку, но плохие: `torch.tensor([...])` из тензоров с `requires_grad` молча копирует значения через `.item()` с предупреждением, собирать надо через `torch.stack` под `no_grad`. И `grad_auto = x.grad` не копия, а тот же тензор: повторный `backward` без обнуления удвоит его. Ниже исправленная версия и дополнительная сверка с центральной разностью из части А

In [5]:
def fixed_grad_check():
    def f(x):
        return torch.sin(x[0]) * x[1]**2 + x[0] * x[2]

    x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
    f(x).backward()
    grad_auto = x.grad.clone()

    with torch.no_grad():
        grad_manual = torch.stack([
            torch.cos(x[0]) * x[1]**2 + x[2],
            2 * torch.sin(x[0]) * x[1],
            x[0],
        ])
    grad_num = numerical_grad_central(f, x.detach().double(), 1e-5).float()
    return torch.allclose(grad_auto, grad_manual, atol=1e-4) and torch.allclose(grad_auto, grad_num, atol=1e-4)


assert fixed_grad_check()

## Уровень 2. Посложнее

### Задача 2. Своя функция для autograd: устойчивый Softplus

Реализуйте класс `StableSoftplus(torch.autograd.Function)`, вычисляющий
$\mathrm{softplus}(x) = \log(1 + e^{x})$ поэлементно, но **численно устойчиво** — то есть без
переполнения `exp` на больших положительных входах. Стандартный приём: для $x > 0$ используйте
тождество $\mathrm{softplus}(x) = x + \log(1 + e^{-x})$ (показатель экспоненты здесь всегда
$\le 0$ и не переполняется), а для $x \le 0$ — прямую формулу $\log(1+e^{x})$, желательно через
`torch.log1p`, которая точнее вычисляет $\log(1+z)$ для маленьких по модулю $z$, чем
`torch.log(1 + z)`.

Вам нужно написать и `forward`, и `backward` вручную (без использования встроенного
`torch.nn.functional.softplus` внутри — иначе задача теряет смысл). Подсказка для `backward`:
производная softplus по $x$ — это в точности сигмоида $\sigma(x) = \dfrac{1}{1+e^{-x}}$.

**Проверьте себя:**
1. `torch.autograd.gradcheck` на вашей функции (в `float64`!) должен проходить.
2. Значение должно совпадать с `torch.nn.functional.softplus` на обычных входах.
3. На входе вида `[1000., -1000., 0.]` ваша функция не должна давать `inf`/`nan` — а наивная
   реализация `torch.log(1 + torch.exp(x))` на таком входе как раз сломается (даст `inf` на
   `1000.`). Продемонстрируйте это явно, сравнив с наивной версией.


In [6]:
class StableSoftplus(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        e = torch.exp(-x.abs())
        ctx.save_for_backward(x, e)
        return x.clamp_min(0) + torch.log1p(e)

    @staticmethod
    def backward(ctx, grad_out):
        x, e = ctx.saved_tensors
        return grad_out * torch.where(x >= 0, 1, e) / (1 + e)


softplus = StableSoftplus.apply

x = (5 * torch.randn(300, dtype=torch.float64)).requires_grad_()
assert torch.autograd.gradcheck(softplus, (x,))
assert torch.allclose(softplus(x), F.softplus(x))

x = torch.tensor([1000., -1000., 0.], requires_grad=True)
naive = torch.log(1 + torch.exp(x))
g_naive, = torch.autograd.grad(naive.sum(), x)
stable = softplus(x)
g_stable, = torch.autograd.grad(stable.sum(), x)

assert torch.isfinite(stable).all() and torch.isfinite(g_stable).all()
assert torch.isinf(naive[0]) and torch.isnan(g_naive[0])
pd.DataFrame({'x': x.detach(), 'naive': naive.detach(), 'stable': stable.detach(),
              'grad naive': g_naive, 'grad stable': g_stable})

,x,naive,stable,grad naive,grad stable
0,1000.0,inf,1000.000000,NaN,1.0
1,-1000.0,0.000000,0.000000,0.0,0.0
2,0.0,0.693147,0.693147,0.5,0.5


Обе ветки из условия сведены в одну формулу без `where`: softplus(x) = max(x, 0) + log1p(e^(−|x|)). При x > 0 это x + log1p(e^(−x)), при x ≤ 0 это log1p(eˣ), а показатель экспоненты всегда ≤ 0

e^(−|x|) сохраняется в forward и переиспользуется в backward, сигмоида собирается из него же без повторной экспоненты: σ(x) = 1/(1 + e) при x ≥ 0 и e/(1 + e) при x < 0, обе дроби не переполняются

Наивная версия ломается не только в значении: на x = 1000 её градиент считается как eˣ/(1 + eˣ) = inf/inf и даёт nan, который при обучении испортил бы все параметры за один шаг

---
**Что сдавать:** ноутбук с реализованными функциями, пройденными проверками (`assert`) и
короткими ответами на вопросы для решённых задач в markdown-ячейках. Все вычисления в этом
задании занимают на CPU не больше пары минут суммарно — если что-то считается заметно дольше,
скорее всего, в решении есть неэффективность (например, Python-цикл там, где нужна векторизация)
или ошибка, приводящая к дивергенции.
